# CLIP-Catalog: GPU Training & Benchmarking Pipeline

This turnkey notebook runs end-to-end on **Kaggle (T4/P100 GPU)** or **Google Colab**:
1. Clones the [`findsamirks-commits/CLIP-Catalog`](https://github.com/findsamirks-commits/CLIP-Catalog) repository
2. Ingests the **Fashion-Product-Images** dataset (44K items, ~45 sub-categories)
3. Evaluates the **Pretrained OpenCLIP ViT-B/32 Zero-Shot Baseline**
4. Runs **InfoNCE Contrastive Fine-Tuning** with Cosine Learning Rate Schedule & AMP
5. Constructs a **FAISS `IndexFlatIP` Exact Cosine Retrieval Index**
6. Outputs the final **Empirical Benchmark Table** (Zero-Shot Top-1, Recall@5, Recall@10, MRR)

## 1. Setup & Environment Verification

In [ ]:
# Verify GPU acceleration
!nvidia-smi

# Clone repository if not already in directory
import os
if not os.path.exists("CLIP-Catalog"):
    !git clone https://github.com/findsamirks-commits/CLIP-Catalog.git
    %cd CLIP-Catalog
else:
    %cd CLIP-Catalog
    !git pull

# Install dependencies
!pip install -q open-clip-torch faiss-cpu pandas pillow pyyaml

## 2. Ingest Catalog Data

In [ ]:
# Check if running inside Kaggle with fashion-product-images-small attached
kaggle_path = "/kaggle/input/fashion-product-images-small/styles.csv"
kaggle_img_dir = "/kaggle/input/fashion-product-images-small/images"

if os.path.exists(kaggle_path):
    print("Found Kaggle Fashion Dataset! Ingesting...")
    !python -m src.data --source fashion --input $kaggle_path --image-dir $kaggle_img_dir --output data/processed/catalog.jsonl
else:
    print("Using high-fidelity synthetic quick-commerce catalog sample (500 SKUs)...")
    !python -m src.data --source synthetic --num-skus 500 --output data/processed/catalog.jsonl

## 3. Pretrained OpenCLIP ViT-B/32 Baseline Evaluation

In [ ]:
print("Evaluating Out-of-the-Box Zero-Shot Baseline...")
!python -m src.eval --catalog data/processed/catalog.jsonl

## 4. Contrastive Fine-Tuning (InfoNCE Loss)

In [ ]:
# Execute fine-tuning with cosine LR schedule
!python -m src.train \
    --config configs/clip_catalog.yaml \
    --catalog data/processed/catalog.jsonl \
    --out checkpoints/clip_catalog_best.pt \
    --epochs 5 \
    --batch-size 64

## 5. Build FAISS Exact Cosine Retrieval Index

In [ ]:
!python -m src.index \
    --model-path checkpoints/clip_catalog_best.pt \
    --catalog data/processed/catalog.jsonl \
    --index-out data/processed/catalog.faiss

## 6. Fine-Tuned Model Benchmark & Relative Lift Report

In [ ]:
print("Running Full Evaluation on Fine-Tuned Checkpoint...")
!python -m src.eval \
    --model-path checkpoints/clip_catalog_best.pt \
    --catalog data/processed/catalog.jsonl

## 7. Interactive Zero-Shot Onboarding Test

In [ ]:
from src.inference import CLIPRetriever
from src.onboarding_bridge import OnboardingBridge

retriever = CLIPRetriever("checkpoints/clip_catalog_best.pt")
bridge = OnboardingBridge(auto_tag_confidence=0.70, review_queue_confidence=0.40, min_top1_margin=0.15)

# Text search test
query = "organic crunchy red apples 500g"
results = retriever.text_to_sku(query, top_k=3)
print(f"\nSearch Results for '{query}':")
for r in results:
    print(f"  Rank #{r['rank']} | Score: {r['score']:.3f} | {r['title']} ({r['category']})")